# 05 — Inspect the Change

This notebook is the **mechanistic companion** to the earlier PEFT notebooks.

The goal is not just to ask *which method worked*, but also:

- **what actually changed inside the model?**
- **where is the adaptation concentrated?**
- **how large are the changes?**
- **what kind of object is being learned?**

We compare four lightweight methods on the **same base checkpoint** and **same dataset**:

1. **Linear probing** — only the classifier head learns  
2. **Prompt tuning** — a learnable image-space prompt changes the input  
3. **Adapters** — a bottleneck module changes the hidden representation  
4. **LoRA** — a low-rank update changes selected weights indirectly

The notebook keeps everything intentionally compact so attendees can run it in Colab and inspect the differences visually.

## What to look for

For each method, we will inspect one or two signatures:

- **Linear probe:** how much the head relies on different frozen features
- **Prompt tuning:** what the learned prompt patch looks like and how much it perturbs the input
- **Adapters:** how large the adapter residual is relative to the frozen representation
- **LoRA:** the size and layer-wise distribution of the low-rank update

This is not meant to be a rigorous interpretability benchmark. It is a **teaching notebook** that helps participants connect:
**method → mechanism → empirical behavior**.

In [ ]:
# If you are in Colab, uncomment:
# !pip install -q torch torchvision matplotlib pandas tqdm peft

import copy
from dataclasses import asdict

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from src.methods.linear_probe import LinearProbeModel
from src.methods.prompt_tuning import PromptTunedClassifier
from src.methods.adapters import AdapterHeadClassifier
from src.methods.lora import apply_lora
from src.training import train_model, count_trainable_parameters

torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Dataset and checkpoint

We use **FashionMNIST** converted to 3 channels so the same minimal image-space prompt code works cleanly.

To keep the workshop runtime short:
- we use a subset of the data
- we train a small frozen backbone once
- all PEFT methods adapt the **same pretrained checkpoint**

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
])

train_ds_full = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_ds_full = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

train_indices = list(range(0, 2500))
val_indices = list(range(0, 800))

train_ds = Subset(train_ds_full, train_indices)
val_ds = Subset(test_ds_full, val_indices)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=2)

num_classes = 10

In [ ]:
class TinyBackbone(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.GELU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.GELU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.proj = nn.Linear(64, feature_dim)

    def forward(self, x):
        h = self.features(x).flatten(1)
        return self.proj(h)


class PretrainClassifier(nn.Module):
    def __init__(self, backbone, feature_dim=128, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))

In [ ]:
# Pretrain once, then freeze the backbone for all downstream PEFT experiments.
feature_dim = 128
base_backbone = TinyBackbone(feature_dim=feature_dim)
pretrain_model = PretrainClassifier(base_backbone, feature_dim=feature_dim, num_classes=num_classes).to(device)

optimizer = torch.optim.Adam(pretrain_model.parameters(), lr=2e-3)
pretrain_history = train_model(pretrain_model, train_loader, val_loader, optimizer, epochs=5, device=device)

frozen_backbone = copy.deepcopy(pretrain_model.backbone).cpu().eval()
for p in frozen_backbone.parameters():
    p.requires_grad = False

pd.DataFrame({
    "epoch": range(1, len(pretrain_history.train_loss)+1),
    "train_loss": pretrain_history.train_loss,
    "val_acc": pretrain_history.val_acc,
})

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(pretrain_history.train_loss)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Backbone pretraining loss")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3.5))
plt.plot(pretrain_history.val_acc)
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Backbone pretraining validation accuracy")
plt.tight_layout()
plt.show()

## Build the PEFT variants

In [ ]:
class LoRAHeadClassifier(nn.Module):
    """Frozen backbone + plain linear head. apply_lora() wraps the head with PEFT LoRA."""
    def __init__(self, backbone, feature_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.head = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)


def build_models(backbone):
    lora_base = LoRAHeadClassifier(copy.deepcopy(backbone), feature_dim, num_classes)
    lora_model = apply_lora(lora_base, target_modules=["head"], rank=8, alpha=16)

    return {
        "linear_probe": LinearProbeModel(copy.deepcopy(backbone), feature_dim, num_classes),
        "prompt_tuning": PromptTunedClassifier(copy.deepcopy(backbone), feature_dim, num_classes, prompt_size=8),
        "adapters": AdapterHeadClassifier(copy.deepcopy(backbone), feature_dim, num_classes, bottleneck_dim=24),
        "lora": lora_model,
    }

In [ ]:
models = build_models(frozen_backbone)

results = []
histories = {}
trained_models = {}

for method_name, model in models.items():
    model = model.to(device)
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=2e-3
    )
    hist = train_model(model, train_loader, val_loader, optimizer, epochs=6, device=device)
    histories[method_name] = hist
    trained_models[method_name] = copy.deepcopy(model).cpu().eval()

    results.append({
        "method": method_name,
        "trainable_params": count_trainable_parameters(model),
        "final_val_acc": hist.val_acc[-1],
        "final_train_loss": hist.train_loss[-1],
    })

results_df = pd.DataFrame(results).sort_values("final_val_acc", ascending=False).reset_index(drop=True)
results_df

In [ ]:
plt.figure(figsize=(7, 4))
for name, hist in histories.items():
    plt.plot(hist.val_acc, label=name)
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Validation accuracy by method")
plt.legend()
plt.tight_layout()
plt.show()

## A small helper batch for inspection

In [ ]:
inspect_x, inspect_y = next(iter(val_loader))
inspect_x = inspect_x[:16]
inspect_y = inspect_y[:16]

with torch.no_grad():
    frozen_feats = frozen_backbone(inspect_x)

frozen_feats.shape

# 1) Linear probing — inspect the learned head

Linear probing does **not** change the backbone at all.
So the most useful thing to inspect is the **classifier head** that maps frozen features to labels.

Questions to ask:
- Are many frozen features used, or only a few?
- Is the head very sparse / concentrated?
- Does the method succeed mostly because the frozen representation was already good?

In [ ]:
linear_model = trained_models["linear_probe"]
head_weight = linear_model.head.weight.detach()  # [num_classes, feature_dim]
feature_importance = head_weight.abs().mean(dim=0)

plt.figure(figsize=(8, 3))
plt.bar(range(len(feature_importance)), feature_importance.numpy())
plt.xlabel("Frozen feature index")
plt.ylabel("Mean |weight| across classes")
plt.title("Linear probe: which frozen features matter?")
plt.tight_layout()
plt.show()

print("Top-10 most used frozen features:", torch.topk(feature_importance, k=10).indices.tolist())

### Interpretation

A linear probe can only **reweight the existing representation**.
If performance is strong here, it suggests the frozen checkpoint already contains a fairly linearly separable signal for the task.

# 2) Prompt tuning — inspect the learned prompt patch

Prompt tuning changes the **input** before the frozen backbone sees it.

In this minimal image setting, that means a small learnable patch is added in the corner of the image.
We can inspect:
- the learned prompt itself
- the average perturbation magnitude it causes

In [ ]:
prompt_model = trained_models["prompt_tuning"]
prompt_patch = prompt_model.prompt.prompt.detach()[0]  # [C, H, W]

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for c in range(3):
    im = axes[c].imshow(prompt_patch[c].numpy())
    axes[c].set_title(f"Prompt channel {c}")
    axes[c].axis("off")
    fig.colorbar(im, ax=axes[c], fraction=0.046, pad=0.04)
plt.suptitle("Learned image-space prompt")
plt.tight_layout()
plt.show()

In [ ]:
with torch.no_grad():
    prompted_x = prompt_model.prompt(inspect_x)
    mean_abs_input_change = (prompted_x - inspect_x).abs().mean().item()

print(f"Mean absolute input perturbation from prompt: {mean_abs_input_change:.6f}")

fig, axes = plt.subplots(2, 6, figsize=(10, 3.5))
for i in range(6):
    axes[0, i].imshow(inspect_x[i].permute(1, 2, 0).mean(dim=-1), cmap="gray")
    axes[0, i].set_title("original")
    axes[0, i].axis("off")

    axes[1, i].imshow(prompted_x[i].permute(1, 2, 0).mean(dim=-1), cmap="gray")
    axes[1, i].set_title("prompted")
    axes[1, i].axis("off")

plt.suptitle("Prompt tuning changes the input seen by the frozen backbone")
plt.tight_layout()
plt.show()

### Interpretation

This is a good visual reminder that prompt methods adapt the model by **steering the input/conditioning pathway** rather than inserting new computation into the hidden layers.

# 3) Adapters — inspect the adapter residual

Adapters keep the backbone frozen, but add a **small trainable hidden-state transformation**.

A useful quantity here is the ratio:

\[
\frac{\|\text{adapter}(h) - h\|}{\|h\|}
\]

where \(h\) is the frozen representation.

This tells us how strongly the adapter is modifying the frozen features.

In [ ]:
adapter_model = trained_models["adapters"]

with torch.no_grad():
    base_feats = adapter_model.backbone(inspect_x)
    adapted_feats = adapter_model.adapter(base_feats)
    residual = adapted_feats - base_feats
    residual_ratio = residual.norm(dim=1) / (base_feats.norm(dim=1) + 1e-8)

print("Mean adapter residual ratio:", residual_ratio.mean().item())
print("Median adapter residual ratio:", residual_ratio.median().item())

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.hist(residual_ratio.numpy(), bins=12)
plt.xlabel("||adapter(h)-h|| / ||h||")
plt.ylabel("count")
plt.title("Adapter strength across inspected examples")
plt.tight_layout()
plt.show()

In [ ]:
# Which hidden features are changed the most on average?
mean_feature_change = residual.abs().mean(dim=0)

plt.figure(figsize=(8, 3))
plt.bar(range(len(mean_feature_change)), mean_feature_change.numpy())
plt.xlabel("Feature index")
plt.ylabel("Mean |adapter residual|")
plt.title("Adapters: where hidden-state changes are concentrated")
plt.tight_layout()
plt.show()

### Interpretation

Adapters act in the **representation / hidden-state domain**.
Compared with prompt tuning, the model is no longer just altering the input—it is explicitly modifying the internal feature vector before classification.

# 4) LoRA — inspect the low-rank weight update

LoRA keeps the base weight frozen and learns a low-rank update:

\[
W' = W + \Delta W, \quad \Delta W = BA \cdot \text{scaling}
\]

So the natural thing to inspect is:
- the magnitude of \(\Delta W\)
- the ratio \(\|\Delta W\| / \|W\|\)
- how much the update changes logits/features on examples

In [ ]:
lora_model = trained_models["lora"]

# PeftModel proxies attribute access to the wrapped model, so lora_model.head
# gives the PEFT LoRA layer that replaced the original nn.Linear.
lora_head = lora_model.head  # peft.tuners.lora.layer.Linear

delta_w = (
    lora_head.lora_B["default"].weight
    @ lora_head.lora_A["default"].weight
) * lora_head.scaling["default"]
base_w = lora_head.base_layer.weight.detach()

print("Base head weight shape:", tuple(base_w.shape))
print("Delta weight shape:", tuple(delta_w.shape))
print("||base W||           =", base_w.norm().item())
print("||delta W||          =", delta_w.norm().item())
print("||delta W|| / ||W||  =", (delta_w.norm() / (base_w.norm() + 1e-8)).item())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
im0 = axes[0].imshow(base_w.numpy(), aspect="auto")
axes[0].set_title("Frozen base head weight")
axes[0].set_xlabel("feature")
axes[0].set_ylabel("class")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(delta_w.detach().numpy(), aspect="auto")
axes[1].set_title("Learned LoRA update ΔW")
axes[1].set_xlabel("feature")
axes[1].set_ylabel("class")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
with torch.no_grad():
    feats = lora_model.backbone(inspect_x)
    logits_base = lora_head.base_layer(feats)
    logits_lora = lora_head(feats)
    logit_shift = (logits_lora - logits_base).norm(dim=1)

plt.figure(figsize=(6, 3.5))
plt.hist(logit_shift.numpy(), bins=12)
plt.xlabel("||logits_lora - logits_base||")
plt.ylabel("count")
plt.title("LoRA-induced output shift on inspected examples")
plt.tight_layout()
plt.show()

print("Mean logit shift:", logit_shift.mean().item())

### Interpretation

LoRA acts in the **weight domain**.
Unlike adapters, which add a hidden-state transformation, LoRA keeps the forward structure nearly unchanged and parameterizes the update as a **low-rank correction** to an existing weight matrix.

# Compare the “inspection signatures” side by side

In [ ]:
inspection_summary = pd.DataFrame([
    {
        "method": "linear_probe",
        "inspection_signal": "head feature concentration",
        "value": float(feature_importance.max().item()),
        "note": "higher max weight concentration = more reliance on a few frozen features",
    },
    {
        "method": "prompt_tuning",
        "inspection_signal": "mean |input perturbation|",
        "value": float(mean_abs_input_change),
        "note": "shows how strongly the prompt changes the input space",
    },
    {
        "method": "adapters",
        "inspection_signal": "mean residual ratio",
        "value": float(residual_ratio.mean().item()),
        "note": "shows how strongly hidden representations are changed",
    },
    {
        "method": "lora",
        "inspection_signal": "||ΔW|| / ||W||",
        "value": float((delta_w.norm() / (base_w.norm() + 1e-8)).item()),
        "note": "shows the size of the learned low-rank weight correction",
    },
])
inspection_summary

# What to use when — now with a mechanistic lens

A useful workshop discussion is to connect the observed mechanism to method choice.

### Linear probing
Use when:
- you want the cheapest strong baseline
- you suspect the frozen representation is already good
- you need simplicity and speed

### Prompt tuning
Use when:
- you like the idea of steering the model through the input/conditioning pathway
- architecture/deployment constraints make prompt-like adaptation attractive
- you want a very lightweight intervention

### Adapters
Use when:
- you want to modify hidden representations directly
- you want a modular trainable block
- you expect the frozen representation needs a learned transformation, not just a new head

### LoRA
Use when:
- you want weight-space adaptation with few trainable parameters
- you want a strong practical compromise between flexibility and efficiency
- you want something closer to “fine-tuning the layer” without updating the full weight

# Suggested attendee exercises

1. **Change the prompt size** from 8 to 4 or 12.  
   How does the learned prompt look? Does the perturbation grow?

2. **Change the adapter bottleneck** from 24 to 8 or 48.  
   Does the residual ratio become smaller or larger?

3. **Change LoRA rank** from 8 to 2 or 16.  
   How does \(\|\Delta W\| / \|W\|\) change? Does accuracy move with it?

4. **Inspect failure cases.**  
   Reduce the training subset to 500 examples and compare which signatures become weaker or noisier.

5. **Relate Notebook 2 to this notebook.**  
   Which methods converged quickly, and do their internal changes look large or surprisingly small?